# 02 - Região e população

Sem população, não dá para comparar lugares. Este notebook traz do IBGE:

- a **grande região** de cada UF (Norte, Nordeste, Centro-Oeste, Sudeste, Sul);
- a **população estimada de 2026** de cada município e UF, o mesmo ano dos dados criminais.

Com isso, criamos duas tabelas novas no mesmo banco, `municipios` e `ufs`. Elas servem a dois propósitos:

- calcular taxas por 100 mil habitantes nas análises;
- permitir que o LLM responda perguntas como "qual região tem a maior taxa de homicídio?".

## 1. Fontes

As duas fontes são APIs públicas do IBGE, consultadas com `requests`:

- **Localidades**: a lista oficial de municípios com código, UF e grande região.
- **Estimativas da População** (agregado 6579, variável 9324): a população residente estimada em 1º de julho de 2026. A estimativa é uma projeção oficial feita a partir do Censo 2022, não uma contagem.

Ao final, salvamos uma cópia em `data/raw` para registrar exatamente o que foi usado.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd
import requests

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

pasta_raw = Path("..") / "data" / "raw"
caminho_banco = Path("..") / "data" / "db" / "crimes.db"

url_localidades = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios?view=nivelado"
url_populacao = "https://servicodados.ibge.gov.br/api/v3/agregados/6579/periodos/2026/variaveis/9324?localidades={nivel}[all]"

In [ ]:
resposta = requests.get(url_localidades, timeout=120)
resposta.raise_for_status()
localidades = pd.DataFrame(resposta.json())
print(localidades.shape)
print(localidades.columns.tolist())
localidades.head(3)

In [ ]:
localidades = localidades.rename(columns={
    "municipio-id": "codigo_ibge",
    "municipio-nome": "nome_ibge",
    "UF-sigla": "uf",
    "UF-nome": "nome_uf",
    "regiao-nome": "regiao",
})[["codigo_ibge", "nome_ibge", "uf", "nome_uf", "regiao"]]
localidades["codigo_ibge"] = localidades["codigo_ibge"].astype(str)
print(localidades.shape)
print(localidades.groupby("regiao")["uf"].agg(ufs="nunique", municipios="size"))

A API de população devolve um JSON aninhado. A função abaixo extrai só o código da localidade e o número de habitantes. O parâmetro `nivel` escolhe o recorte: `N6` é município e `N3` é UF.

In [ ]:
def baixar_populacao(nivel):
    resposta = requests.get(url_populacao.format(nivel=nivel), timeout=120)
    resposta.raise_for_status()
    series = resposta.json()[0]["resultados"][0]["series"]
    return pd.DataFrame({
        "codigo_ibge": [s["localidade"]["id"] for s in series],
        "populacao": [int(s["serie"]["2026"]) for s in series],
    })


populacao_municipios = baixar_populacao("N6")
print(populacao_municipios.shape)
print("População total:", f"{populacao_municipios['populacao'].sum():,}")
populacao_municipios.head()

In [ ]:
ibge = localidades.merge(populacao_municipios, on="codigo_ibge", how="outer", indicator=True)
print(ibge["_merge"].value_counts())
ibge = ibge.drop(columns="_merge")
print(ibge.shape)
ibge.head()

Conferência: a soma dos municípios de cada UF deve bater com a população que o IBGE publica para a UF.

In [ ]:
populacao_ufs = baixar_populacao("N3")
codigo_uf = ibge["codigo_ibge"].str[:2]
soma_municipios = ibge.groupby(codigo_uf)["populacao"].sum()
conferencia = populacao_ufs.set_index("codigo_ibge").join(soma_municipios.rename("soma_municipios"))
conferencia["diferenca"] = conferencia["populacao"] - conferencia["soma_municipios"]
print("UFs:", len(conferencia))
print("UFs com diferença:", (conferencia["diferenca"] != 0).sum())

In [ ]:
caminho_ibge = pasta_raw / "ibge_municipios_populacao_2026.csv"
ibge.to_csv(caminho_ibge, index=False)
print("Salvo em", caminho_ibge, ibge.shape)

## 2. Ligar o IBGE aos municípios do banco

O banco não tem código IBGE, só a UF e o nome em maiúsculas. Então a ligação é pelo par `(uf, nome)`, nunca só pelo nome, por causa dos homônimos vistos no notebook 01.

A junção `outer` mostra os dois lados do que não casou:

- `left_only`: nomes que estão no banco e não no IBGE;
- `right_only`: nomes que estão no IBGE e não no banco.

In [ ]:
conexao = sqlite3.connect(caminho_banco)
banco = pd.read_sql("SELECT DISTINCT uf, municipio FROM ocorrencias WHERE municipio != 'NÃO INFORMADO'", conexao)
print("Municípios no banco:", len(banco))

ibge["municipio"] = ibge["nome_ibge"].str.upper().str.strip()
juncao = banco.merge(ibge, on=["uf", "municipio"], how="outer", indicator=True)
print(juncao["_merge"].value_counts())
juncao.loc[juncao["_merge"] != "both", ["uf", "municipio", "_merge", "populacao"]].sort_values(["uf", "municipio"])

As diferenças são de grafia, sempre dentro da mesma UF. Por exemplo, o Sinesp escreve "AÇU" e o IBGE escreve "ASSÚ".

Corrigimos com um dicionário explícito, que leva o nome do IBGE ao nome do Sinesp. Isso é melhor do que uma comparação "aproximada" automática, que poderia ligar municípios errados sem avisar.

Boa Esperança do Norte (MT) existe no IBGE mas não aparece no banco. É um município recente, e não verificamos onde o Sinesp registra as ocorrências dele. A população dele continua contando para o total de MT.

In [ ]:
correcoes = {
    ("MG", "BARÃO DO MONTE ALTO"): "BARÃO DE MONTE ALTO",
    ("MT", "SANTO ANTÔNIO DE LEVERGER"): "SANTO ANTÔNIO DO LEVERGER",
    ("RN", "AREZ"): "ARÊS",
    ("RN", "ASSÚ"): "AÇU",
    ("RR", "SÃO LUIZ DO ANAUÁ"): "SÃO LUIZ",
    ("SE", "GRACCHO CARDOSO"): "GRACHO CARDOSO",
}
for (uf, nome_ibge), nome_banco in correcoes.items():
    linha = (ibge["uf"] == uf) & (ibge["municipio"] == nome_ibge)
    print(uf, nome_ibge, "->", nome_banco, "| linhas:", linha.sum())
    ibge.loc[linha, "municipio"] = nome_banco

In [ ]:
juncao = banco.merge(ibge, on=["uf", "municipio"], how="outer", indicator=True)
print(juncao["_merge"].value_counts())
juncao.loc[juncao["_merge"] != "both", ["uf", "municipio", "nome_ibge", "_merge", "populacao"]]

## 3. Gravar as tabelas no banco

Criamos duas tabelas novas no banco:

- **`municipios`**: uma linha por município do IBGE, com `uf` e `municipio` escritos como no Sinesp, para a junção com `ocorrencias`. Tem também o nome oficial, o código IBGE, a região e a população;
- **`ufs`**: uma linha por UF, com nome, região e população (soma dos municípios, que já conferimos contra o total do IBGE).

O `carga.py` só substitui a tabela `ocorrencias`, então essas duas tabelas continuam no banco se a carga for refeita.

In [ ]:
municipios = ibge[["uf", "municipio", "nome_ibge", "codigo_ibge", "regiao", "populacao"]].sort_values(["uf", "municipio"])
ufs = ibge.groupby(["uf", "nome_uf", "regiao"], as_index=False)["populacao"].sum()
print(municipios.shape, ufs.shape)
print("Pares (uf, municipio) duplicados:", municipios.duplicated(["uf", "municipio"]).sum())
ufs

In [ ]:
municipios.to_sql("municipios", conexao, if_exists="replace", index=False)
ufs.to_sql("ufs", conexao, if_exists="replace", index=False)
conexao.execute("CREATE UNIQUE INDEX idx_municipios_uf_municipio ON municipios (uf, municipio)")
conexao.commit()
print(pd.read_sql("SELECT name, type FROM sqlite_master ORDER BY type, name", conexao))
print(pd.read_sql("SELECT COUNT(*) AS municipios, SUM(populacao) AS populacao FROM municipios", conexao))
print(pd.read_sql("SELECT COUNT(*) AS ufs, SUM(populacao) AS populacao FROM ufs", conexao))

## 4. Primeiro uso: taxa por 100 mil habitantes

A taxa é calculada assim:

`taxa = casos / população × 100.000`

Ela responde: "a cada 100 mil moradores, quantos casos houve?". Assim, uma UF grande e uma pequena ficam na mesma escala.

Duas observações:

- A taxa abaixo cobre **jan a out (10 meses)**. Não é anual, então não é comparável diretamente com taxas anuais publicadas em outros lugares.
- Diferenças entre UFs podem vir tanto de diferenças reais quanto de diferenças na forma de registrar.

A consulta junta `ocorrencias` com `ufs` pela coluna `uf` e filtra uma única abrangência, como definido no notebook 01.

In [ ]:
sql_uf = '''
SELECT u.regiao, o.uf, SUM(o.quantidade) AS vitimas, u.populacao,
       ROUND(SUM(o.quantidade) * 100000.0 / u.populacao, 2) AS taxa_100mil
FROM ocorrencias o
JOIN ufs u ON u.uf = o.uf
WHERE o.evento = 'Homicídio doloso' AND o.abrangencia = 'Estadual'
GROUP BY o.uf
ORDER BY taxa_100mil DESC
'''
taxa_uf = pd.read_sql(sql_uf, conexao)
print(taxa_uf.shape, "| vítimas:", taxa_uf["vitimas"].sum())
taxa_uf

Comparação entre as posições no ranking por contagem bruta e por taxa. Se as duas colunas fossem parecidas, a população não faria diferença.

In [ ]:
taxa_uf["posicao_bruta"] = taxa_uf["vitimas"].rank(ascending=False, method="min").astype(int)
taxa_uf["posicao_taxa"] = taxa_uf["taxa_100mil"].rank(ascending=False, method="min").astype(int)
taxa_uf[["uf", "vitimas", "posicao_bruta", "taxa_100mil", "posicao_taxa"]].sort_values("posicao_bruta")

Por região, há uma armadilha no SQL. Se juntássemos `ocorrencias` com `ufs` e somássemos `populacao` direto, a população de cada UF seria contada uma vez para cada linha de ocorrência. A taxa sairia errada sem nenhum erro aparecer.

A solução é somar as vítimas por UF primeiro (o bloco `WITH`) e só depois juntar com a população. Essa regra vai entrar no prompt do LLM.

In [ ]:
sql_regiao = '''
WITH vitimas_uf AS (
    SELECT uf, SUM(quantidade) AS vitimas
    FROM ocorrencias
    WHERE evento = 'Homicídio doloso' AND abrangencia = 'Estadual'
    GROUP BY uf
)
SELECT u.regiao, SUM(v.vitimas) AS vitimas, SUM(u.populacao) AS populacao,
       ROUND(SUM(v.vitimas) * 100000.0 / SUM(u.populacao), 2) AS taxa_100mil
FROM vitimas_uf v
JOIN ufs u ON u.uf = v.uf
GROUP BY u.regiao
ORDER BY taxa_100mil DESC
'''
taxa_regiao = pd.read_sql(sql_regiao, conexao)
print("Vítimas:", taxa_regiao["vitimas"].sum(), "| População:", f"{taxa_regiao['populacao'].sum():,}")
taxa_regiao

In [ ]:
sql_errado = '''
SELECT u.regiao, SUM(o.quantidade) AS vitimas, SUM(u.populacao) AS populacao_somada_errada
FROM ocorrencias o
JOIN ufs u ON u.uf = o.uf
WHERE o.evento = 'Homicídio doloso' AND o.abrangencia = 'Estadual'
GROUP BY u.regiao
'''
pd.read_sql(sql_errado, conexao)

In [ ]:
conexao.close()

## 5. Resumo

O que os prints acima mostram:

- **IBGE**
  - 5.571 municípios em 27 UFs e 5 regiões. A população estimada de 2026 soma 214.211.951 pessoas.
  - A soma dos municípios bate com a população publicada para cada UF, sem diferença em nenhuma das 27.
- **Ligação com o banco**
  - Pelo par (uf, nome), 5.564 municípios casaram direto.
  - Os 6 restantes eram diferenças de grafia e foram corrigidos com um dicionário explícito. Depois disso, os 5.570 municípios do banco têm população.
  - Boa Esperança do Norte (MT) está no IBGE e não aparece no banco.
- **Tabelas novas no banco**
  - `municipios` tem 5.571 linhas e liga com `ocorrencias` por `(uf, municipio)`.
  - `ufs` tem 27 linhas, com nome, região e população.
- **Contagem bruta x taxa**
  - Em homicídio doloso (Estadual, jan a out), as posições mudam muito entre os dois critérios.
  - SP é a 4ª UF em vítimas e a 27ª em taxa. AP é a 25ª em vítimas e a 5ª em taxa.
- **Por região**
  - A taxa de jan a out vai de 17,80 (Nordeste) a 6,30 (Sul) por 100 mil habitantes.
  - Esses números descrevem o registro. Não explicam a causa da diferença, e parte dela pode vir de diferenças na forma de registrar entre as UFs.
- **Armadilha de SQL**
  - Somar a população depois de juntar com `ocorrencias` multiplica a população pelo número de linhas.
  - É preciso agregar os casos por UF antes de juntar.

Regras novas para os próximos notebooks e para o prompt do LLM:

- a taxa é `casos * 100000.0 / populacao`, e o período (10 meses) precisa ser dito junto;
- a junção com `ufs` ou `municipios` só pode ser feita depois de agregar `ocorrencias`;
- município é sempre o par `(uf, municipio)`.